# **2. Geração do VCF com Parsnp e HarvestTools**

Este notebook prepara o conjunto de genomas utilizado no alinhamento de core genome e gera o VCF que servirá de entrada para a construção das matrizes de genótipos.

O conjunto analisado possui duas origens:

1. **Genomas provenientes do NCBI**, previamente avaliados e aprovados no notebook
   `01_acquire_and_qc_dataset.ipynb`;
2. **Genomas fornecidos diretamente pelo laboratório**, armazenados separadamente como
   dados de entrada manuais.

A organização relevante dos dados é:

```text
data/
├── raw/
│   └── v4/                     # snapshot original obtido do NCBI
│
├── manual/
│   └── incoming/               # FASTAs recebidos diretamente do laboratório
│
├── reference/                  # recursos de referência do projeto
│
├── work/
│   └── v4/
│       ├── ncbi_approved/      # materialização dos genomas NCBI aprovados
│       └── parsnp_inputs.txt   # lista de FASTAs utilizados pelo Parsnp
│
└── results/
    └── v4/
        └── parsnp/             # resultados do alinhamento
```

O notebook `01_acquire_and_qc_dataset.ipynb` também produz um manifesto contendo exatamente os assemblies do NCBI aprovados após o controle de qualidade. Esse manifesto é versionado junto ao projeto e funciona como o contrato entre as etapas de aquisição/QC e de alinhamento:

```text
clade-clustering/
└── manifests/
    └── v4/
        └── approved_genomes.csv
```

Há um manifesto independente para cada versão do dataset. Dessa forma, os arquivos genômicos pesados não precisam ser versionados nem transferidos entre ambientes: o manifesto registra quais assemblies devem compor o conjunto, e este notebook os materializa diretamente no ambiente em que o Parsnp será executado.

A partir do manifesto, os genomas aprovados são recuperados com o `NCBI Datasets CLI`. Os FASTAs resultantes são mantidos em data/work/v4/ncbi_approved/. Os genomas
recebidos do laboratório permanecem em `data/manual/incoming/`.

As duas fontes são então reunidas logicamente em `parsnp_inputs.txt`, que contém os caminhos dos FASTAs utilizados no alinhamento. Não há cópia ou reorganização física dos
genomas para um único diretório.

O assembly `GCA_003013715.2 (B11220)` é utilizado como genoma de referência e é fornecido separadamente ao Parsnp. Os demais genomas são utilizados como conjunto de entrada para o alinhamento.

O fluxo deste notebook é, portanto:

```text
approved_genomes.csv
        │
        ▼
materialização dos genomas NCBI
        │
        ├───────────────┐
        │               │
        ▼               ▼
genomas NCBI      genomas manuais
        │               │
        └───────┬───────┘
                ▼
        parsnp_inputs.txt
                │
                ▼
             Parsnp
                │
           parsnp.ggr
                │
                ▼
         HarvestTools
                │
                ▼
           parsnp.vcf
```

O arquivo parsnp.vcf produzido ao final desta etapa será processado no notebook `03_build_snp_matrices.ipynb`, responsável pela construção das matrizes de genótipos utilizadas posteriormente nas análises de Machine Learning.

## **2.1 Imports**

In [39]:
import zipfile
import shutil
import subprocess
import shlex

from pathlib import Path

import pandas as pd

from clade_clustering.paths import (
    MANIFESTS_DIR,
    MANUAL_DIR,
    WORK_DIR,
    RESULTS_DIR,
)

## **2.2 Dataset configs**

In [3]:
DATA_VERSION = "v4"

APPROVED_GENOMES_MANIFEST_PATH = (
    MANIFESTS_DIR / DATA_VERSION / "approved_genomes.csv"
)

MANUAL_INCOMING_DIR = MANUAL_DIR / "incoming"

WORK_VERSION_DIR = WORK_DIR / DATA_VERSION
PARSNP_INPUT_DIR = WORK_VERSION_DIR / "parsnp_input"

In [4]:
for path in [
    MANUAL_INCOMING_DIR,
    PARSNP_INPUT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Manifesto: {APPROVED_GENOMES_MANIFEST_PATH}")
print(f"Manual:    {MANUAL_INCOMING_DIR}")
print(f"Parsnp:    {PARSNP_INPUT_DIR}")

Manifesto: /workspace/projects/clade-clustering/manifests/v4/approved_genomes.csv
Manual:    /workspace/projects/clade-clustering/data/manual/incoming
Parsnp:    /workspace/projects/clade-clustering/data/work/v4/parsnp_input


In [23]:
DATASETS_BIN = shutil.which("datasets")
PARSNP_BIN = shutil.which("parsnp")
HARVESTTOOLS_BIN = shutil.which("harvesttools")

executables = {
    "datasets": DATASETS_BIN,
    "parsnp": PARSNP_BIN,
    "harvesttools": HARVESTTOOLS_BIN,
}

missing = [
    name for name, path in executables.items()
    if path is None
]

if missing:
    raise RuntimeError(
        f"Executáveis não encontrados: {missing}"
    )

if not APPROVED_GENOMES_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifesto não encontrado:\n"
        f"{APPROVED_GENOMES_MANIFEST_PATH}"
    )

manifest = pd.read_csv(APPROVED_GENOMES_MANIFEST_PATH)

print(f"Genomas aprovados no manifesto: {len(manifest)}")

for name, path in executables.items():
    print(f"{name:12s}: {path}")

Genomas aprovados no manifesto: 1244
datasets    : /opt/conda/bin/datasets
parsnp      : /opt/conda/bin/parsnp
harvesttools: /opt/conda/bin/harvesttools


## **2.3 Download the large dataset of approved genomes: dehydrate & rehydrate**

Os `accessions` aprovados no notebook 01 são materializados diretamente no ambiente de execução a partir do manifesto versionado. Para conjuntos com mais de **1000** assemblies, utiliza-se o fluxo dehydrated/rehydrate do NCBI Datasets: https://www.ncbi.nlm.nih.gov/datasets/docs/v2/how-tos/genomes/large-download

### **2.3.1 Building directories to store the approved genomes**

In [8]:
NCBI_WORK_DIR = WORK_VERSION_DIR / "ncbi_approved"

ACCESSIONS_PATH = NCBI_WORK_DIR / "approved_accessions.txt"
NCBI_DEHYDRATED_ZIP = NCBI_WORK_DIR / "approved_genomes_dehydrated.zip"
NCBI_PACKAGE_DIR = NCBI_WORK_DIR / "package"


NCBI_WORK_DIR.mkdir(parents=True, exist_ok=True)

### **2.3.2 Validation of the manifest**

Validação do arquivo de manifesto seguido da criação de `approved_accessions.txt`:

In [17]:
required_columns = {"accession", "fasta_filename"}

missing_columns = required_columns - set(manifest.columns)

if missing_columns:
    raise RuntimeError(
        f"Colunas ausentes no manifesto: {sorted(missing_columns)}"
    )

if manifest["accession"].isna().any():
    raise RuntimeError(
        "O manifesto contém accessions ausentes."
    )

if manifest["accession"].duplicated().any():
    raise RuntimeError(
        "O manifesto contém accessions duplicados."
    )

accessions = manifest["accession"].astype(str)

In [ ]:
accessions.to_csv(
    ACCESSIONS_PATH,
    index=False,
    header=False,
)

print(f"Accessions preparados: {len(accessions)}")
print(f"Arquivo: {ACCESSIONS_PATH}")

Accessions preparados: 1244
Arquivo: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_accessions.txt


Validação adicional de contrato:

In [19]:
with open(ACCESSIONS_PATH) as f:
    accession_lines = [
        line.strip()
        for line in f
        if line.strip()
    ]

print(f"Linhas no arquivo: {len(accession_lines)}")
print(accession_lines[:5])

if accession_lines != accessions.tolist():
    raise RuntimeError(
        "O arquivo de accessions não corresponde ao manifesto."
    )

Linhas no arquivo: 1244
['GCA_001189475.1', 'GCA_002759435.3', 'GCA_002775015.1', 'GCA_003013715.2', 'GCA_003014415.1']


### **2.3.3 Download and extraction of the zip file for approved genomes**

In [24]:
if NCBI_DEHYDRATED_ZIP.exists():
    if not zipfile.is_zipfile(NCBI_DEHYDRATED_ZIP):
        raise RuntimeError(
            f"O arquivo existente não é um ZIP válido:\n"
            f"{NCBI_DEHYDRATED_ZIP}"
        )

    print(
        "Pacote dehydrated já existe. "
        "Download será ignorado."
    )

else:
    cmd = [
        DATASETS_BIN,
        "download",
        "genome",
        "accession",
        "--inputfile",
        str(ACCESSIONS_PATH),
        "--include",
        "genome",
        "--dehydrated",
        "--filename",
        str(NCBI_DEHYDRATED_ZIP),
    ]

    print("Gerando pacote dehydrated do NCBI...")

    subprocess.run(
        cmd,
        check=True,
    )

    if not NCBI_DEHYDRATED_ZIP.exists():
        raise RuntimeError(
            "O pacote dehydrated não foi criado."
        )

    if not zipfile.is_zipfile(NCBI_DEHYDRATED_ZIP):
        raise RuntimeError(
            "O pacote baixado não é um ZIP válido."
        )

    print(
        f"Pacote salvo em:\n"
        f"{NCBI_DEHYDRATED_ZIP}"
    )

Gerando pacote dehydrated do NCBI...


New version of client (18.39.0) available at https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/LATEST/linux-amd64/datasets.
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    847B 7.85MB/s
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    847B 7.85MB/s
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    847B 7.85MB/s
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    847B 7.85MB/s
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    847B 7.85MB/s
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    847B 7.85MB/s
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    847

Pacote salvo em:
/workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip


Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    364kB 132kB/s
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    364kB 132kB/s
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    364kB 132kB/s
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    364kB 132kB/s
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    364kB 132kB/s
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    364kB 132kB/s
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    364kB 132kB/s
Downloading: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/approved_genomes_dehydrated.zip    364kB 

In [25]:
if not NCBI_DEHYDRATED_ZIP.exists():
    raise FileNotFoundError(
        f"Pacote dehydrated não encontrado:\n"
        f"{NCBI_DEHYDRATED_ZIP}"
    )

if not zipfile.is_zipfile(NCBI_DEHYDRATED_ZIP):
    raise RuntimeError(
        f"O pacote dehydrated não é um ZIP válido:\n"
        f"{NCBI_DEHYDRATED_ZIP}"
    )

size_mb = NCBI_DEHYDRATED_ZIP.stat().st_size / 1e6

with zipfile.ZipFile(NCBI_DEHYDRATED_ZIP, "r") as zf:
    members = zf.namelist()

print(f"Tamanho do ZIP: {size_mb:.2f} MB")
print(f"Entradas no ZIP: {len(members)}")
print("Primeiras entradas:")

for member in members[:10]:
    print(member)

Tamanho do ZIP: 0.43 MB
Entradas no ZIP: 5
Primeiras entradas:
README.md
ncbi_dataset/data/assembly_data_report.jsonl
ncbi_dataset/fetch.txt
ncbi_dataset/data/dataset_catalog.json
md5sum.txt


In [26]:
if NCBI_PACKAGE_DIR.exists() and any(NCBI_PACKAGE_DIR.iterdir()):
    raise RuntimeError(
        f"O diretório de extração já contém arquivos:\n"
        f"{NCBI_PACKAGE_DIR}"
    )

NCBI_PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(NCBI_DEHYDRATED_ZIP, "r") as zf:
    zf.extractall(NCBI_PACKAGE_DIR)

extracted_files = [
    path
    for path in NCBI_PACKAGE_DIR.rglob("*")
    if path.is_file()
]

print(f"Arquivos extraídos: {len(extracted_files)}")
print(f"Pacote extraído em: {NCBI_PACKAGE_DIR}")

Arquivos extraídos: 5
Pacote extraído em: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/package


In [27]:
FETCH_PATH = NCBI_PACKAGE_DIR / "ncbi_dataset" / "fetch.txt"

if not FETCH_PATH.exists():
    raise FileNotFoundError(
        f"fetch.txt não encontrado:\n{FETCH_PATH}"
    )

with open(FETCH_PATH) as f:
    fetch_lines = [
        line.strip()
        for line in f
        if line.strip()
    ]

print(f"Entradas em fetch.txt: {len(fetch_lines)}")
print("Primeiras entradas:")
for line in fetch_lines[:5]:
    print(line)

Entradas em fetch.txt: 1244
Primeiras entradas:
https://api.ncbi.nlm.nih.gov/datasets/fetch_h/R2V0UmVtb3RlRGF0YWZpbGU/eNqTyufKzUtOytROKymw0tdPT83Lz00t1k_MydF3d3bUNzAw1De0sNQ3MTcF8eOBfCAXyNMzjHcM9oVwygzxycWDzcxM1kvLS9RLrzJgtGAEAH1EH9c	0	data/GCA_001189475.1/GCA_001189475.1_ASM118947v1_genomic.fna
https://api.ncbi.nlm.nih.gov/datasets/fetch_h/R2V0UmVtb3RlRGF0YWZpbGU/eNqTamTkqs9LTsrUTispsNLXT0_Ny89NLdZPzMnRd3d21DcwMNI3NzfVNzA0BfHjgXwgF8jTM4x3TsxLiU8sLcosjncyNDQyMowPMyRKUTzYlsxkvbS8RL30KgNGC0YAc7snRg	0	data/GCA_002775015.1/GCA_002775015.1_Cand_auris_B11221_V1_genomic.fna
https://api.ncbi.nlm.nih.gov/datasets/fetch_h/R2V0UmVtb3RlRGF0YWZpbGU/eNqTyufKzUtOytROKymw0tdPT83Lz00t1k_MydF3d3bUNzAw1jcwNNY3NzQF8eOBfCAXyNMzincM9oVwyozwycWDzcxM1kvLS9RLrzJgtGAEAG7pH5A	0	data/GCA_003013715.2/GCA_003013715.2_ASM301371v2_genomic.fna
https://api.ncbi.nlm.nih.gov/datasets/fetch_h/R2V0UmVtb3RlRGF0YWZpbGU/eNqTquaqzEtOytROKymw0tdPT83Lz00t1k_MydF3d3bUNzAw1jcwNNE3MTQF8eOBfCAXyNMzjHdOzEuJTywtyiyOdzI0NDIxJqwiHmx-ZrJ

Agora o processo rehydrate: https://www.ncbi.nlm.nih.gov/datasets/docs/v2/reference-docs/command-line/datasets/datasets_rehydrate/

In [28]:
cmd = [
    DATASETS_BIN,
    "rehydrate",
    "--directory",
    str(NCBI_PACKAGE_DIR),
]

print("Reidratando genomas aprovados no Sirius...")

subprocess.run(
    cmd,
    check=True,
)

print("Reidratação concluída.")

Reidratando genomas aprovados no Sirius...
Found 1244 of 1244 files for rehydration
Completed 0 of 1244 [------------------------------------------------]   0%
Completed 0 of 1244 [------------------------------------------------]   0%
Completed 0 of 1244 [------------------------------------------------]   0%
Completed 0 of 1244 [------------------------------------------------]   0%
Completed 0 of 1244 [------------------------------------------------]   0%
Completed 0 of 1244 [------------------------------------------------]   0%
Completed 0 of 1244 [------------------------------------------------]   0%
Completed 0 of 1244 [------------------------------------------------]   0%
Completed 0 of 1244 [------------------------------------------------]   0%
Completed 0 of 1244 [------------------------------------------------]   0%
Completed 0 of 1244 [------------------------------------------------]   0%
Completed 0 of 1244 [------------------------------------------------]   0%
Comp

In [29]:
NCBI_DATA_DIR = (
    NCBI_PACKAGE_DIR / "ncbi_dataset" / "data"
)

downloaded_fastas = sorted(
    NCBI_DATA_DIR.rglob("*_genomic.fna")
)

downloaded_accessions = {
    path.parent.name
    for path in downloaded_fastas
}

manifest_accessions = set(
    manifest["accession"].astype(str)
)

missing_accessions = sorted(
    manifest_accessions - downloaded_accessions
)

unexpected_accessions = sorted(
    downloaded_accessions - manifest_accessions
)

print(f"FASTAs reidratados: {len(downloaded_fastas)}")
print(f"Accessions únicos reidratados: {len(downloaded_accessions)}")
print(f"Accessions no manifesto: {len(manifest_accessions)}")
print(f"Ausentes: {len(missing_accessions)}")
print(f"Não esperados: {len(unexpected_accessions)}")

FASTAs reidratados: 1244
Accessions únicos reidratados: 1244
Accessions no manifesto: 1244
Ausentes: 0
Não esperados: 0


In [30]:
if len(downloaded_fastas) != len(manifest):
    raise RuntimeError(
        "O número de FASTAs reidratados não corresponde ao manifesto."
    )

if len(downloaded_accessions) != len(downloaded_fastas):
    raise RuntimeError(
        "Há mais de um FASTA associado a pelo menos um accession."
    )

if missing_accessions or unexpected_accessions:
    raise RuntimeError(
        "O conjunto de accessions materializado não corresponde "
        "exatamente ao manifesto."
    )

print("Conjunto NCBI materializado e validado com sucesso.")

Conjunto NCBI materializado e validado com sucesso.


### **2.3.4 Identifying manually added genomes**

Além dos genomas NCBI definidos pelo manifesto, o pipeline pode incorporar
genomas fornecidos diretamente pelo laboratório. Esses arquivos são mantidos
separadamente dos snapshots NCBI e adicionados posteriormente à entrada do Parsnp.

No `v4`, são 11 FASTAs versionados no repositório em `data/manual/incoming/`: seis genomas do laboratório (Ca9–Ca14) e cinco montagens de corridas SRA (`SRR*.pilon.fasta`). Eles entram no alinhamento, mas são reservados como hold-out nos notebooks 04 e 05.

In [32]:
FASTA_SUFFIXES = {".fna", ".fa", ".fasta"}

manual_fastas = sorted(
    path
    for path in MANUAL_INCOMING_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in FASTA_SUFFIXES
)

print(f"FASTAs manuais encontrados: {len(manual_fastas)}")

for path in manual_fastas[:10]:
    print(path.name)

if len(manual_fastas) > 10:
    print(f"... e mais {len(manual_fastas) - 10}")

FASTAs manuais encontrados: 11
Ca10.fasta
Ca11.fasta
Ca12.fasta
Ca13.fasta
Ca14.fasta
Ca9.fasta
SRR18325429.pilon.fasta
SRR18325430.pilon.fasta
SRR18325431.pilon.fasta
SRR18325432.pilon.fasta
... e mais 1


In [33]:
manual_names = [path.name for path in manual_fastas]

if len(manual_names) != len(set(manual_names)):
    raise RuntimeError(
        "Há FASTAs manuais com nomes de arquivo duplicados."
    )

In [ ]:
EXPECTED_MANUAL_FASTAS = {
    # Genomas do laboratório
    "Ca9.fasta",
    "Ca10.fasta",
    "Ca11.fasta",
    "Ca12.fasta",
    "Ca13.fasta",
    "Ca14.fasta",
    # Montagens Pilon de corridas SRA
    "SRR9007776.pilon.fasta",
    "SRR18325429.pilon.fasta",
    "SRR18325430.pilon.fasta",
    "SRR18325431.pilon.fasta",
    "SRR18325432.pilon.fasta",
}

missing_manual = sorted(EXPECTED_MANUAL_FASTAS - set(manual_names))
unexpected_manual = sorted(set(manual_names) - EXPECTED_MANUAL_FASTAS)

if missing_manual:
    raise RuntimeError(
        f"FASTAs manuais ausentes em {MANUAL_INCOMING_DIR}:\n"
        + "\n".join(f"- {name}" for name in missing_manual)
        + "\nEsses arquivos são versionados no repositório; "
        "verifique se o clone está completo."
    )

if unexpected_manual:
    raise RuntimeError(
        f"FASTAs manuais não esperados em {MANUAL_INCOMING_DIR}:\n"
        + "\n".join(f"- {name}" for name in unexpected_manual)
        + "\nOs notebooks 04 e 05 esperam exatamente os 6 genomas Ca "
        "e os 5 SRR."
    )

print(f"Genomas manuais esperados presentes: {len(EXPECTED_MANUAL_FASTAS)}")

In [34]:
ncbi_names = {path.name for path in downloaded_fastas}

name_collisions = sorted(
    ncbi_names & set(manual_names)
)

if name_collisions:
    raise RuntimeError(
        "Há nomes de FASTA presentes tanto no conjunto NCBI "
        f"quanto no conjunto manual: {name_collisions[:10]}"
    )

print("Entradas manuais validadas.")

Entradas manuais validadas.


## **2.5 Parsnp**

### **2.5.1 Build the Parsnp input: NCBI genomes + manual**

```text
data/work/v4/
├── ncbi_approved/
│   ├── approved_accessions.txt
│   ├── approved_genomes_dehydrated.zip
│   └── package/
│       └── ncbi_dataset/data/.../*.fna
│
└── parsnp_inputs.txt
```

```text
NCBI reidratado ─┐
                 ├──> parsnp_inputs.txt ───> Parsnp
manual/incoming ─┘
```

In [35]:
PARSNP_INPUT_LIST_PATH = (
    WORK_VERSION_DIR / "parsnp_inputs.txt"
)

input_fastas = downloaded_fastas + manual_fastas

input_names = [path.name for path in input_fastas]

if len(input_names) != len(set(input_names)):
    raise RuntimeError(
        "Há nomes de FASTA duplicados no conjunto final."
    )

with open(PARSNP_INPUT_LIST_PATH, "w") as f:
    for path in input_fastas:
        f.write(f"{path.resolve()}\n")

print(f"FASTAs NCBI: {len(downloaded_fastas)}")
print(f"FASTAs manuais: {len(manual_fastas)}")
print(f"Entradas Parsnp: {len(input_fastas)}")
print(f"Lista salva em: {PARSNP_INPUT_LIST_PATH}")

FASTAs NCBI: 1244
FASTAs manuais: 11
Entradas Parsnp: 1255
Lista salva em: /workspace/projects/clade-clustering/data/work/v4/parsnp_inputs.txt


In [36]:
with open(PARSNP_INPUT_LIST_PATH) as f:
    parsnp_inputs = [
        Path(line.strip())
        for line in f
        if line.strip()
    ]

missing_inputs = [
    path
    for path in parsnp_inputs
    if not path.exists()
]

if len(parsnp_inputs) != len(input_fastas):
    raise RuntimeError(
        "A lista de entrada não corresponde ao conjunto esperado."
    )

if missing_inputs:
    raise RuntimeError(
        f"Há arquivos ausentes na lista: {missing_inputs[:10]}"
    )

print("Lista de entrada do Parsnp validada.")

Lista de entrada do Parsnp validada.


### **2.5.2 Define the reference genome**

O alinhamento com Parsnp utiliza como referência o assembly B11220,
correspondente ao accession `GCA_003013715.2`.

In [37]:
REFERENCE_ACCESSION = "GCA_003013715.2"

reference_candidates = [
    path
    for path in downloaded_fastas
    if path.parent.name == REFERENCE_ACCESSION
]

if len(reference_candidates) != 1:
    raise RuntimeError(
        f"Esperado exatamente um FASTA para {REFERENCE_ACCESSION}, "
        f"mas foram encontrados {len(reference_candidates)}."
    )

REFERENCE_FASTA = reference_candidates[0]

print(f"Referência: {REFERENCE_ACCESSION}")
print(f"FASTA: {REFERENCE_FASTA}")

Referência: GCA_003013715.2
FASTA: /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/package/ncbi_dataset/data/GCA_003013715.2/GCA_003013715.2_ASM301371v2_genomic.fna


In [38]:
if REFERENCE_FASTA.resolve() not in {
    path.resolve() for path in parsnp_inputs
}:
    raise RuntimeError(
        "O genoma de referência não está presente "
        "na lista de entrada do Parsnp."
    )

print("Genoma de referência validado.")

Genoma de referência validado.


### **2.5.3 Run the Parsnp tool**

In [44]:
input_fastas = [
    path
    for path in downloaded_fastas
    if path.resolve() != REFERENCE_FASTA.resolve()
] + manual_fastas

with open(PARSNP_INPUT_LIST_PATH, "w") as f:
    for path in input_fastas:
        f.write(f"{path.resolve()}\n")

print(f"Genomas de entrada do Parsnp: {len(input_fastas)}")
print(f"Referência separada: {REFERENCE_FASTA.name}")

Genomas de entrada do Parsnp: 1254
Referência separada: GCA_003013715.2_ASM301371v2_genomic.fna


In [45]:
with open(PARSNP_INPUT_LIST_PATH) as f:
    parsnp_inputs = [
        Path(line.strip())
        for line in f
        if line.strip()
    ]

if REFERENCE_FASTA.resolve() in {
    path.resolve() for path in parsnp_inputs
}:
    raise RuntimeError(
        "A referência não deve aparecer entre os genomas de -d."
    )

if len(parsnp_inputs) != len(input_fastas):
    raise RuntimeError(
        "A lista de entrada do Parsnp está inconsistente."
    )

if any(not path.exists() for path in parsnp_inputs):
    raise RuntimeError(
        "Há FASTAs ausentes na lista de entrada."
    )

print("Entradas não referência validadas.")

Entradas não referência validadas.


In [54]:
PARSNP_OUTPUT_DIR = RESULTS_DIR / DATA_VERSION / "parsnp"

PARSNP_OUTPUT_DIR.parent.mkdir(
    parents=True,
    exist_ok=True,
)

PARSNP_THREADS = 64

In [55]:
PARSNP_OUTPUT_DIR

PosixPath('/workspace/projects/clade-clustering/data/results/v4/parsnp')

In [56]:
parsnp_cmd = [
    PARSNP_BIN,
    "-d",
    str(PARSNP_INPUT_LIST_PATH),
    "-r",
    str(REFERENCE_FASTA),
    "-o",
    str(PARSNP_OUTPUT_DIR),
    "-p",
    str(PARSNP_THREADS),
]


print("Comando Parsnp:")
print(shlex.join(parsnp_cmd))

Comando Parsnp:
/opt/conda/bin/parsnp -d /workspace/projects/clade-clustering/data/work/v4/parsnp_inputs.txt -r /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/package/ncbi_dataset/data/GCA_003013715.2/GCA_003013715.2_ASM301371v2_genomic.fna -o /workspace/projects/clade-clustering/data/results/v4/parsnp -p 64


In [57]:
if PARSNP_OUTPUT_DIR.exists():
    raise RuntimeError(
        f"O diretório de saída já existe:\n{PARSNP_OUTPUT_DIR}\n"
        "Verifique-o antes de executar novamente o Parsnp."
    )

print("Executando Parsnp...")
print(shlex.join(parsnp_cmd))

subprocess.run(
    parsnp_cmd,
    check=True,
)

print("Parsnp concluído com sucesso.")

Executando Parsnp...
/opt/conda/bin/parsnp -d /workspace/projects/clade-clustering/data/work/v4/parsnp_inputs.txt -r /workspace/projects/clade-clustering/data/work/v4/ncbi_approved/package/ncbi_dataset/data/GCA_003013715.2/GCA_003013715.2_ASM301371v2_genomic.fna -o /workspace/projects/clade-clustering/data/results/v4/parsnp -p 64


01:04:00 - INFO - |--Parsnp 2.1.5--|

01:04:01 - INFO - 
****************************************************************************
SETTINGS:
|-refgenome:	/workspace/projects/clade-clustering/data/work/v4/ncbi_approved/package/ncbi_dataset/data/GCA_003013715.2/GCA_003013715.2_ASM301371v2_genomic.fna
|-genomes:	
	/workspace/projects/clade-clustering/data/work/v4/ncbi_approved/package/ncbi_dataset/data/GCA_001189475.1/GCA_001189475.1_ASM118947v1_genomic.fna
	/workspace/projects/clade-clustering/data/work/v4/ncbi_approved/package/ncbi_dataset/data/GCA_002759435.3/GCA_002759435.3_Cand_auris_B8441_V3_genomic.fna
	...1250 more file(s)...
	/workspace/projects/clade-clustering/data/manual/incoming/SRR18325432.pilon.fasta
	/workspace/projects/clade-clustering/data/manual/incoming/SRR9007776.pilon.fasta
|-aligner:	muscle
|-outdir:	/workspace/projects/clade-clustering/data/results/v4/parsnp
|-OS:	Linux
|-threads:	64
****************************************************************************
  

Parsnp concluído com sucesso.


02:28:59 - INFO - Aligned 1255 genomes in 1.42 hours
02:28:59 - INFO - Parsnp finished! All output available in /workspace/projects/clade-clustering/data/results/v4/parsnp


- **Validação dos outputs do Parsnp:**

In [58]:
parsnp_outputs = sorted(
    path
    for path in PARSNP_OUTPUT_DIR.iterdir()
    if path.is_file()
)

print("Arquivos produzidos pelo Parsnp:")
for path in parsnp_outputs:
    size_mb = path.stat().st_size / 1e6
    print(f"{path.name:25s} {size_mb:10.2f} MB")

Arquivos produzidos pelo Parsnp:
GCA_003013715.2_ASM301371v2_genomic.fna.ref      12.25 MB
parsnp.ggr                    134.74 MB
parsnp.maf                   6051.08 MB
parsnp.snps.mblocks           153.65 MB
parsnp.tree                     0.11 MB
parsnp.xmfa                  5587.63 MB
parsnpAligner.log               0.00 MB


In [60]:
PARSNP_GGR_PATH = PARSNP_OUTPUT_DIR / "parsnp.ggr"
PARSNP_VCF_PATH = PARSNP_OUTPUT_DIR / "parsnp.vcf"

if not PARSNP_GGR_PATH.exists():
    raise FileNotFoundError(
        f"parsnp.ggr não encontrado:\n{PARSNP_GGR_PATH}"
    )

if PARSNP_GGR_PATH.stat().st_size == 0:
    raise RuntimeError(
        "parsnp.ggr existe, mas está vazio."
    )

print(f"GGR validado: {PARSNP_GGR_PATH}")

GGR validado: /workspace/projects/clade-clustering/data/results/v4/parsnp/parsnp.ggr


### **2.5.4 Run the HarvestTools**

In [61]:
harvesttools_cmd = [
    HARVESTTOOLS_BIN,
    "-i",
    str(PARSNP_GGR_PATH),
    "-V",
    str(PARSNP_VCF_PATH),
]

print("Comando HarvestTools:")
print(shlex.join(harvesttools_cmd))

Comando HarvestTools:
/opt/conda/bin/harvesttools -i /workspace/projects/clade-clustering/data/results/v4/parsnp/parsnp.ggr -V /workspace/projects/clade-clustering/data/results/v4/parsnp/parsnp.vcf


In [62]:
if PARSNP_VCF_PATH.exists():
    raise RuntimeError(
        f"O VCF de saída já existe:\n{PARSNP_VCF_PATH}\n"
        "Verifique-o antes de executar novamente o HarvestTools."
    )

print("Gerando VCF com HarvestTools...")

subprocess.run(
    harvesttools_cmd,
    check=True,
)

if not PARSNP_VCF_PATH.exists():
    raise RuntimeError(
        "HarvestTools terminou sem produzir parsnp.vcf."
    )

if PARSNP_VCF_PATH.stat().st_size == 0:
    raise RuntimeError(
        "parsnp.vcf foi criado, mas está vazio."
    )

print("VCF gerado com sucesso.")
print(f"Arquivo: {PARSNP_VCF_PATH}")
print(
    f"Tamanho: "
    f"{PARSNP_VCF_PATH.stat().st_size / 1e6:.2f} MB"
)

Gerando VCF com HarvestTools...


Loading /workspace/projects/clade-clustering/data/results/v4/parsnp/parsnp.ggr...
Writing /workspace/projects/clade-clustering/data/results/v4/parsnp/parsnp.vcf...


VCF gerado com sucesso.
Arquivo: /workspace/projects/clade-clustering/data/results/v4/parsnp/parsnp.vcf
Tamanho: 431.57 MB
